# 04 — Daily-Horizon IC Analysis (Phase 2C)

**Pivot from Phase 3.** The 15-min combos (combo1/2/4/5) all failed baseline backtest because IC ≈ 0.04 at 15-min horizon translates to per-trade edge < 10 bp roundtrip cost on SPY. The fix isn't tuning parameters — it's changing horizon. At daily/weekly holds the per-trade gross edge scales with the holding period while cost stays constant.

This notebook re-runs IC analysis at **daily** frequency with **daily-appropriate indicators**: momentum (5d/21d/63d/12-1), MA distance, RSI, money-flow (OBV/MFI/CMF), volume z-score, realized vol, ATR z-score. Forward horizons: 1d, 5d, 21d, 63d (one quarter).

Universe: SPY, QQQ, IWM. Sample: 2020-01 → 2024-12 (5 years, covers COVID crash + recovery + 2022 bear + 2023-24 rally).

Implementation: `daily_indicators.py` (copied here per Option A; canonical at `../analysis/daily_indicators.py`).

In [ ]:
# region imports
from AlgorithmImports import *
from QuantConnect.Research import QuantBook

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from daily_indicators import compute_all, ALL as INDICATOR_FNS
from ic_calculator import compute_ic, ic_verdict, quantile_test, rolling_ic
# endregion

qb = QuantBook()
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 140)

TICKERS = ['SPY', 'QQQ', 'IWM']
PERIODS = (1, 5, 21, 63)      # days: 1, 1wk, 1mo, 1qtr
END   = datetime(2024, 12, 31)
START = END - timedelta(days=365 * 5)

## 1. Fetch daily data

5 years × 3 symbols × 1 bar/day ≈ 3,800 bars total. Trivial fetch — should be near-instant.

In [ ]:
symbols = {t: qb.add_equity(t, Resolution.DAILY).symbol for t in TICKERS}
history = qb.history(list(symbols.values()), START, END, Resolution.DAILY)
print(f'daily rows: {len(history):,}')

bars = {t: history.loc[sym] for t, sym in symbols.items()}
for t, df in bars.items():
    print(f'  {t}: {len(df):,} daily bars  ({df.index.min().date()} → {df.index.max().date()})')

bars['SPY'].head(3)

## 2. Compute indicators on every symbol

In [ ]:
indicators_by_sym = {t: compute_all(df) for t, df in bars.items()}
returns_by_sym    = {t: df['close'].pct_change() for t, df in bars.items()}

print('Indicator columns:', list(INDICATOR_FNS.keys()))
print()
print('SPY indicators (last 3 rows, non-null):')
indicators_by_sym['SPY'].dropna().tail(3)

## 3. IC at multiple horizons, per symbol

Spearman rank IC, periods in **days**. 1=next day, 5=1 week, 21=1 month, 63=1 quarter.

In [ ]:
records = []
for sym, ind_df in indicators_by_sym.items():
    rets = returns_by_sym[sym]
    for name in ind_df.columns:
        for period, r in compute_ic(ind_df[name], rets, periods=PERIODS).items():
            records.append({'symbol': sym, 'indicator': name, 'period': period,
                            'ic': r.ic, 'pvalue': r.pvalue, 'n': r.n})
ic_long = pd.DataFrame(records)

for sym in TICKERS:
    table = (ic_long[ic_long.symbol == sym]
             .pivot_table(values='ic', index='indicator', columns='period')
             .reindex(INDICATOR_FNS.keys()))
    print(f'\n── {sym} ── IC by horizon (rows=indicator, cols=days ahead) ──')
    print(table.to_string())

## 4. Averaged IC across symbols + verdict

In [ ]:
avg_ic = (ic_long.groupby(['indicator', 'period'])['ic']
          .mean().unstack('period').reindex(INDICATOR_FNS.keys()))
print('Mean IC across SPY/QQQ/IWM (rows=indicator, cols=days ahead):')
print(avg_ic.to_string())
print()

best_p  = avg_ic.abs().idxmax(axis=1)
best_ic = pd.Series({n: avg_ic.loc[n, best_p[n]] for n in avg_ic.index})
best = pd.DataFrame({
    'best_period': best_p,
    'best_ic':     best_ic,
    'verdict':     best_ic.apply(ic_verdict),
})
print('Best-horizon summary:')
print(best.to_string())

## 5. Quantile tests for survivors

Forward 21-day (≈1 month) returns binned by indicator value. Real signals show monotonic quantile means; flukes are noisy.

In [ ]:
survivors = best[best.verdict != 'reject'].index.tolist()
print(f'survivors: {survivors}\n')

for name in survivors:
    print(f'── {name} — quantile test (period=21d, ≈ 1 month) ──')
    for sym in TICKERS:
        q = quantile_test(indicators_by_sym[sym][name], returns_by_sym[sym],
                          n_quantiles=5, period=21)
        print(f'  {sym}:  ' + '  '.join(
            f'Q{i}={q.loc[i,"mean"]:+.4f}' for i in q.index
        ))

## 6. Regime split — bull vs bear days

Define regime by SPY's trailing 60-day return: bull if > +5%, bear if < −5%, else range. Compute IC for survivors in each regime. A signal that only works in one regime is still useful — but you need to know which.

Limitations: with only 5 years, bear sample is small (mostly 2022). Treat as exploratory.

In [ ]:
spy_60d = bars['SPY']['close'].pct_change(60)
regime = pd.Series(index=spy_60d.index, dtype='object')
regime[spy_60d > 0.05] = 'bull'
regime[spy_60d < -0.05] = 'bear'
regime[(spy_60d >= -0.05) & (spy_60d <= 0.05)] = 'range'
print(f'regime day counts: {regime.value_counts().to_dict()}')
print()

if survivors:
    rows = []
    for sym, ind_df in indicators_by_sym.items():
        rets = returns_by_sym[sym]
        sym_regime = regime.reindex(ind_df.index)
        for name in survivors:
            for r in ('bull', 'bear', 'range'):
                mask = (sym_regime == r)
                if mask.sum() < 30:
                    continue
                ic_r = compute_ic(
                    ind_df[name].where(mask),
                    rets.where(mask),
                    periods=(21,),
                )[21]
                rows.append({
                    'symbol': sym, 'indicator': name, 'regime': r,
                    'ic_21d': ic_r.ic, 'n': ic_r.n,
                })
    regime_ic = pd.DataFrame(rows)
    print('IC at 21d horizon, per regime (mean across symbols):')
    print(regime_ic.pivot_table('ic_21d', index='indicator', columns='regime').to_string())
else:
    print('No survivors — skipping regime split.')

## 7. Rolling 6-month IC stability (SPY only)

Sign-stable rolling IC = the signal works across periods. Sign-flipping = regime-dependent and harder to trade.

In [ ]:
if survivors:
    fig, axes = plt.subplots(len(survivors), 1, figsize=(11, 2.2 * len(survivors)), squeeze=False)
    rets_spy = returns_by_sym['SPY']
    for ax, name in zip(axes[:, 0], survivors):
        ric = rolling_ic(indicators_by_sym['SPY'][name], rets_spy, window=126, period=21)
        ax.plot(ric.index, ric.values, linewidth=0.9)
        ax.axhline(0, color='red', alpha=0.5, linewidth=0.8)
        ax.set_title(f'SPY rolling 6-month IC — {name} (period=21d)', fontsize=10)
        ax.set_ylim(-0.4, 0.4)
    plt.tight_layout()
else:
    print('No survivors — skipping rolling IC plot.')

## 8. Translating IC into per-trade bp edge

Phase 3 lesson: |IC| × σ(returns) × √holding_period ≈ expected per-trade gross return. A daily IC of 0.05 at 21-day hold on SPY (daily σ ≈ 1%):

    edge ≈ 0.05 × 0.01 × √21 × 10000 ≈ 23 bp per trade

Roundtrip cost on SPY ≈ 6-10 bp (IBKR commission + 1bp slippage × 2 + GST). **Daily holds give us ~13 bp net margin if the IC is real.** That's the structural reason this pivot should work where 15-min didn't.

Use the table below as a quick filter: any survivor whose `expected_bp_per_trade − 10` is positive is a Phase 3C candidate.

In [ ]:
if survivors:
    spy_daily_std = returns_by_sym['SPY'].std()
    rows = []
    for name in survivors:
        ic_val = best.loc[name, 'best_ic']
        period = best.loc[name, 'best_period']
        bp = abs(ic_val) * spy_daily_std * np.sqrt(period) * 10000
        rows.append({
            'indicator': name,
            'best_ic':   ic_val,
            'period_d':  period,
            'est_bp_per_trade': round(bp, 1),
            'net_vs_10bp_cost':  round(bp - 10, 1),
        })
    print('Per-trade edge estimate (daily σ_SPY ≈ {:.4f}):'.format(spy_daily_std))
    print(pd.DataFrame(rows).set_index('indicator').to_string())
else:
    print('No survivors.')

## 9. Phase 3C strategy candidates

Survivors with positive `net_vs_10bp_cost` in section 8 are the cost-viable signals. Among those, prefer:

- **Monotonic quantile means** (section 5)
- **Sign-stable rolling IC** (section 7)
- **Works in multiple regimes** (section 6) — or at least clearly works in one

Each surviving signal can become a Phase 3C strategy: enter when indicator in top-quantile, hold the indicated period, exit at horizon or stop. Compared with Phase 3, the structural changes are:

- Daily bars instead of 15-min → 100x fewer trades, costs no longer dominant
- Overnight holds permitted → captures SPY drift instead of fighting it
- Long/short permitted → can profit in 2022-style bear regime